In [ ]:
# Setup
import math_toolkit
math_toolkit.activate()


# Fourier compression of an image

A gray-scale image can be treated as a function on a two-dimensional periodic grid. Its Fourier series has coefficients

$$
c_{m,n}, \qquad -\frac{H}{2}\le m < \frac{H}{2}, \quad -\frac{W}{2}\le n < \frac{W}{2},
$$

where $H$ and $W$ are the image height and width. Large low-frequency coefficients describe broad shapes. Small high-frequency coefficients describe fine detail.


## Load the cat

The file `bw-cat-1.bmp` is a small 16-bit RGB565 bitmap. The loader below reads that BMP format directly so the notebook works in JupyterLite without installing Pillow.


In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def load_rgb565_bmp(path):
    """Load an uncompressed 16-bit RGB565 BMP as floating RGB values."""

    data = Path(path).read_bytes()
    if data[:2] != b"BM":
        raise ValueError("Expected a BMP file.")

    # Read the fixed BMP/DIB header fields used by this notebook's image.
    pixel_offset = int.from_bytes(data[10:14], "little")
    width = int.from_bytes(data[18:22], "little", signed=True)
    signed_height = int.from_bytes(data[22:26], "little", signed=True)
    planes = int.from_bytes(data[26:28], "little")
    bits_per_pixel = int.from_bytes(data[28:30], "little")
    compression = int.from_bytes(data[30:34], "little")

    if planes != 1 or bits_per_pixel != 16 or compression != 3:
        raise ValueError("This loader expects an uncompressed 16-bit RGB565 BMP.")

    # BMP rows are padded to a four-byte boundary. Positive heights store rows
    # from bottom to top, so we reverse them after decoding.
    height = abs(signed_height)
    row_stride = ((bits_per_pixel * width + 31) // 32) * 4
    rows = []

    for row_index in range(height):
        start = pixel_offset + row_index * row_stride
        stop = start + width * 2
        packed = np.frombuffer(data[start:stop], dtype="<u2").astype(np.uint16)

        red = ((packed & 0xF800) >> 11) / 31.0
        green = ((packed & 0x07E0) >> 5) / 63.0
        blue = (packed & 0x001F) / 31.0
        rows.append(np.stack((red, green, blue), axis=-1))

    if signed_height > 0:
        rows.reverse()

    return np.stack(rows, axis=0)


def show_gray_image(image, title, *, width=520, height=560):
    """Display a gray-scale image with square pixels."""

    fig = go.Figure(
        go.Heatmap(
            z=image,
            colorscale="gray",
            zmin=0,
            zmax=1,
            showscale=False,
        )
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(autorange="reversed", scaleanchor="x", showticklabels=False)
    fig.update_layout(title=title, width=width, height=height, margin=dict(l=10, r=10, t=45, b=10))
    fig.show()


cat_rgb = load_rgb565_bmp("bw-cat-1.bmp")
cat = (0.2126 * cat_rgb[:, :, 0] + 0.7152 * cat_rgb[:, :, 1] + 0.0722 * cat_rgb[:, :, 2]).astype(float)

md(f"Loaded `bw-cat-1.bmp` as a `{cat.shape[0]} x {cat.shape[1]}` gray-scale array.")
show_gray_image(cat, "Original image")


## Compute the Fourier series coefficients

`numpy.fft.fft2` gives the two-dimensional Fourier series coefficients on the image grid. We divide by the number of pixels so that `coeffs[m, n]` is the coefficient in the usual averaged Fourier-series normalization.


In [ ]:
height, width = cat.shape
coeffs = np.fft.fft2(cat) / cat.size

# Shift the zero-frequency coefficient to the center for a readable picture.
abs_coeffs = np.abs(np.fft.fftshift(coeffs))
log_abs_coeffs = np.log10(abs_coeffs + 1e-8)

fig = go.Figure(
    go.Heatmap(
        z=log_abs_coeffs,
        colorscale="Viridis",
        colorbar=dict(title="log10 |c_mn|"),
    )
)
fig.update_xaxes(title="horizontal frequency n", showticklabels=False)
fig.update_yaxes(title="vertical frequency m", showticklabels=False)
fig.update_layout(
    title="Absolute values of Fourier coefficients",
    width=640,
    height=640,
    margin=dict(l=55, r=20, t=55, b=55),
)
fig.show()


## Truncate the series

Now keep only the low-frequency diamond

$$
|m|+|n| < N.
$$

Increase `cutoff_N` to preserve more detail. Decrease it to store fewer coefficients.


In [ ]:
# You can change it here.
cutoff_N = 100


In [ ]:
vertical_frequencies = np.fft.fftfreq(height) * height
horizontal_frequencies = np.fft.fftfreq(width) * width
m_grid, n_grid = np.meshgrid(vertical_frequencies, horizontal_frequencies, indexing="ij")

# The retained modes are enough to reconstruct a low-pass version of the image.
kept_mask = np.abs(m_grid) + np.abs(n_grid) < cutoff_N
truncated_coeffs = np.where(kept_mask, coeffs, 0)
reconstructed = np.fft.ifft2(truncated_coeffs * cat.size).real
reconstructed = np.clip(reconstructed, 0, 1)
fourier_error = np.abs(cat - reconstructed)

kept_count = int(kept_mask.sum())
total_count = int(kept_mask.size)
real_value_budget = 2 * kept_count
print(f"Keeping {kept_count:,} of {total_count:,} coefficients ({kept_count / total_count:.2%}) with |m| + |n| < {cutoff_N}.")
print(f"Those coefficients store {real_value_budget:,} real float values: one real and one imaginary part per coefficient.")

# Use the same real-valued storage budget for direct gray samples. The rectangular
# grid is chosen to preserve the image aspect ratio, so the actual count may be
# a few samples below the target.
coarse_height = max(1, int(round(np.sqrt(real_value_budget * height / width))))
coarse_width = max(1, int(round(real_value_budget / coarse_height)))
while coarse_height * coarse_width > real_value_budget and coarse_width > 1:
    coarse_width -= 1

sample_rows = np.linspace(0, height - 1, coarse_height)
sample_cols = np.linspace(0, width - 1, coarse_width)
row_indices = np.rint(sample_rows).astype(int)
col_indices = np.rint(sample_cols).astype(int)
downsampled = cat[np.ix_(row_indices, col_indices)]

# Resize the sampled image back to the original grid with nearest-neighbor lookup.
nearest_rows = np.abs(np.arange(height)[:, None] - sample_rows[None, :]).argmin(axis=1)
nearest_cols = np.abs(np.arange(width)[:, None] - sample_cols[None, :]).argmin(axis=1)
downsample_reconstructed = downsampled[nearest_rows, :][:, nearest_cols]
downsample_error = np.abs(cat - downsample_reconstructed)
downsample_count = int(downsampled.size)

print(f"Downsampling stores {downsample_count:,} gray samples on a {coarse_height} x {coarse_width} grid.")
print(f"Mean Fourier error:    {fourier_error.mean():.4f}")
print(f"Mean downsample error: {downsample_error.mean():.4f}")

fig = make_subplots(
    rows=1,
    cols=5,
    subplot_titles=("Original", "Fourier", "Fourier error", "Downsample", "Downsample error"),
)
fig.add_trace(go.Heatmap(z=cat, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=reconstructed, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=fourier_error, colorscale="Inferno", zmin=0, zmax=float(fourier_error.max()), showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=downsample_reconstructed, colorscale="gray", zmin=0, zmax=1, showscale=False), row=1, col=4)
fig.add_trace(go.Heatmap(z=downsample_error, colorscale="Inferno", zmin=0, zmax=float(downsample_error.max()), showscale=True), row=1, col=5)

for axis_index in range(1, 6):
    fig.update_xaxes(showticklabels=False, row=1, col=axis_index)
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=axis_index)

fig.update_layout(width=1500, height=390, margin=dict(l=10, r=10, t=55, b=10))
fig.show()

## Size comparison

The fair comparison below uses the same `float32` precision for both methods. A stored `complex64` Fourier coefficient contains two `float32` values, so `K` Fourier coefficients have the same scalar precision budget as about `2K` downsampled gray values.

The `uint8` downsample size is shown only as a quantized-image reference, not as the equal-precision comparison.

In [ ]:
bmp_bytes = Path("bw-cat-1.bmp").stat().st_size
gray_uint8_bytes = cat.size * np.dtype(np.uint8).itemsize
gray_float32_bytes = cat.size * np.dtype(np.float32).itemsize
full_complex64_bytes = coeffs.size * np.dtype(np.complex64).itemsize
truncated_complex64_bytes = kept_count * np.dtype(np.complex64).itemsize
downsample_float32_bytes = downsample_count * np.dtype(np.float32).itemsize
downsample_uint8_bytes = downsample_count * np.dtype(np.uint8).itemsize
metadata_bytes = 5 * np.dtype(np.int32).itemsize
fourier_stored_bytes = truncated_complex64_bytes + metadata_bytes
downsample_float32_stored_bytes = downsample_float32_bytes + metadata_bytes
downsample_uint8_stored_bytes = downsample_uint8_bytes + metadata_bytes

print("Data-size comparison")
print(f"  BMP file on disk:                         {bmp_bytes:,} bytes")
print(f"  Decoded 8-bit gray image:                  {gray_uint8_bytes:,} bytes")
print(f"  Decoded float32 gray image:                {gray_float32_bytes:,} bytes")
print(f"  Full complex64 Fourier array:              {full_complex64_bytes:,} bytes")
print(f"  Retained Fourier coefficients:             {kept_count:,} complex64 values")
print(f"  Fourier real-value budget:                 {real_value_budget:,} float32 values")
print(f"  Downsampled gray values:                   {downsample_count:,} float32 values")
print(f"  Truncated Fourier coefficients:            {truncated_complex64_bytes:,} bytes")
print(f"  Equal-precision downsampled float32 image: {downsample_float32_bytes:,} bytes")
print(f"  Quantized downsampled uint8 reference:     {downsample_uint8_bytes:,} bytes")
print(f"  Metadata estimate:                         {metadata_bytes:,} bytes")
print(f"  Total Fourier representation:              {fourier_stored_bytes:,} bytes")
print(f"  Total downsampled float32 image:           {downsample_float32_stored_bytes:,} bytes")
print(f"  Total downsampled uint8 reference:         {downsample_uint8_stored_bytes:,} bytes")
print(f"  Fourier versus float32 gray image:         {fourier_stored_bytes / gray_float32_bytes:.2%}")
print(f"  Downsample versus float32 gray image:      {downsample_float32_stored_bytes / gray_float32_bytes:.2%}")
print(f"  Fourier versus BMP file:                   {fourier_stored_bytes / bmp_bytes:.2%}")
print(f"  Downsample float32 versus BMP file:        {downsample_float32_stored_bytes / bmp_bytes:.2%}")

:::{admonition} Question
Try `cutoff_N = 10`, `20`, `60`, and `100`. Where do you first recognize the cat? Where does the stored data stop feeling small?
:::
